In [0]:
%pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.6/196.6 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.9/542.9 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import col
import pandas as pd

# Load silver master
silver_path = "/Volumes/workspace/default/olist_raw/silver/"
silver = spark.read.format("delta").load(silver_path + "master")

# Get reviews with text only — sample 1000 rows for speed on Community Edition
reviews_sample = silver \
    .select("order_id", "review_score", "review_comment_message") \
    .filter(col("review_comment_message").isNotNull()) \
    .limit(1000) \
    .toPandas()

print(f"Reviews sample: {len(reviews_sample)} rows")
reviews_sample.head(3)

Reviews sample: 1000 rows


,order_id,review_score,review_comment_message
0,d887b52c6516beb39e8cd44a5f8b60f7,5,Ficou tudo ok só pensei q fosse maior mas.ta bom
1,66e4624ae69e7dc89bd50222b59f581f,1,"Pedir 2 capinhas,estou pagando as duas e só ve..."
2,c228d309379c3ae545cc6ca2d5aae1b2,5,"Veio muito rápido, obrigado!!"


In [0]:
from transformers import pipeline

# Load multilingual sentiment analysis model
# This model works on Portuguese text
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=512
)

print("Model loaded successfully")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-24536ecb-6766-40b1-a440-7c2c5bbe456e/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded successfully


In [0]:
# Run sentiment analysis on each review
def analyze_sentiment(text):
    try:
        result = sentiment_pipeline(text[:512])[0]
        return result['label'], round(result['score'], 4)
    except:
        return "error", 0.0

# Apply to all 1000 reviews
reviews_sample[['sentiment_label', 'sentiment_score']] = reviews_sample['review_comment_message'].apply(
    lambda x: pd.Series(analyze_sentiment(str(x)))
)

print("Sentiment analysis complete")
reviews_sample.head(10)

Sentiment analysis complete


,order_id,review_score,review_comment_message,sentiment_label,sentiment_score
0,d887b52c6516beb39e8cd44a5f8b60f7,5,Ficou tudo ok só pensei q fosse maior mas.ta bom,4 stars,0.4233
1,66e4624ae69e7dc89bd50222b59f581f,1,"Pedir 2 capinhas,estou pagando as duas e só ve...",1 star,0.8177
2,c228d309379c3ae545cc6ca2d5aae1b2,5,"Veio muito rápido, obrigado!!",5 stars,0.6070
3,7e05645577366863a93350aac5cc7de5,3,"Recebi o produto, porém o mesmo veio com o enc...",2 stars,0.3848
4,4207ad8e0e4e4070f271c930d25f72d7,1,"Gostei do produto, porém entregaram apenas 1 u...",1 star,0.3752
5,7845a2492ab1b4f2cf3d56c7b8da1446,5,chegou antes do prometido,1 star,0.3497
6,f6f6a68d78c7dd4b319f6df3c846094b,1,Não passou o prazo ainda.,1 star,0.5000
7,03f83b59e7c7355647462745b18119f1,3,"O produto em si é bom, mas a entrega deixou mu...",3 stars,0.4641
8,a2fc0d6559cd85b95e1a02c8d97d55ec,5,Comprei em um dia e recebi no outro!,5 stars,0.4978
9,689624d719c9fffa2c4ff8f71da712e4,4,Bom,5 stars,0.4902


In [0]:
from pyspark.sql.functions import count, avg, round

# Convert back to Spark DataFrame and save
reviews_with_sentiment = spark.createDataFrame(reviews_sample)
gold_path = "/Volumes/workspace/default/olist_raw/gold/"
reviews_with_sentiment.write.format("delta").mode("overwrite").save(gold_path + "reviews_sentiment")

# Summary — how well does sentiment align with review score
summary = reviews_with_sentiment \
    .groupBy("review_score", "sentiment_label") \
    .agg(count("*").alias("count")) \
    .orderBy("review_score", "sentiment_label")

print("Sentiment analysis saved to gold layer")
summary.show(20)

Sentiment analysis saved to gold layer
+------------+---------------+-----+
|review_score|sentiment_label|count|
+------------+---------------+-----+
|           1|         1 star|  140|
|           1|        2 stars|   18|
|           1|        3 stars|   10|
|           1|        4 stars|    1|
|           1|        5 stars|   10|
|           2|         1 star|   27|
|           2|        2 stars|   11|
|           2|        3 stars|    6|
|           3|         1 star|   37|
|           3|        2 stars|   19|
|           3|        3 stars|   19|
|           3|        4 stars|    7|
|           3|        5 stars|   12|
|           4|         1 star|   18|
|           4|        2 stars|   13|
|           4|        3 stars|   30|
|           4|        4 stars|   31|
|           4|        5 stars|   57|
|           5|         1 star|   42|
|           5|        2 stars|   19|
+------------+---------------+-----+
only showing top 20 rows
